# Matchbook Odds Lookup & Price History

Search for a Matchbook exchange match by **home and away team name**, retrieve all associated market and runner definitions, and extract granular odds tick history filtered by market (e.g. `Correct Score`) and runner (e.g. `1-0`), ordered by event timestamp.

### Prerequisites
Ensure Matchbook bronze events (`data/bronze/matchbook_events/`) and odds ticks (`data/bronze/matchbook_odds/`) have been populated via ingestion runs or data migration.

In [ ]:
import json
import os
import re
from pathlib import Path

import duckdb
import pandas as pd

# Locate data directory (works inside container /app/data or locally in ./data)
DATA_DIR = Path(os.environ.get("DATA_DIR", "/app/data" if Path("/app/data").exists() else "data"))
EVENTS_GLOB = f"{DATA_DIR}/bronze/matchbook_events/**/*.parquet"
ODDS_GLOB = f"{DATA_DIR}/bronze/matchbook_odds/**/*.parquet"

# Initialize DuckDB connection
con = duckdb.connect()
print(f"Data directory: {DATA_DIR}")

In [ ]:
def lookup_matchbook_match(home_team: str, away_team: str) -> pd.DataFrame:
    """Look up a Matchbook match by home and away team names.

    Searches across event_name in bronze matchbook_events Parquet files using
    case-insensitive substring matching.
    """
    home_pat = f"%{home_team.strip().lower()}%"
    away_pat = f"%{away_team.strip().lower()}%"

    query = f"""
        SELECT
            event_id,
            event_name,
            sport_id,
            start_utc,
            status,
            max(ingested_at) as latest_ingested_at,
            any_value(raw_event) as raw_event
        FROM read_parquet('{EVENTS_GLOB}', union_by_name=true)
        WHERE lower(event_name) LIKE '{home_pat}'
          AND lower(event_name) LIKE '{away_pat}'
        GROUP BY event_id, event_name, sport_id, start_utc, status
        ORDER BY start_utc DESC
    """
    try:
        df = con.execute(query).df()
        return df
    except Exception as exc:
        print(f"Lookup error (check if bronze event files exist at {EVENTS_GLOB}): {exc}")
        return pd.DataFrame()

In [ ]:
def get_event_markets_and_runners(raw_event_json: str | dict) -> pd.DataFrame:
    """Parse raw_event JSON to extract a catalogue of available markets and runners."""
    if isinstance(raw_event_json, str):
        try:
            payload = json.loads(raw_event_json)
        except Exception:
            return pd.DataFrame()
    else:
        payload = raw_event_json or {}

    rows = []
    event_id = str(payload.get("id", ""))
    event_name = str(payload.get("name", ""))

    for market in payload.get("markets", []):
        market_id = str(market.get("id", ""))
        market_name = str(market.get("name", ""))
        market_type = str(market.get("market-type", ""))
        market_status = str(market.get("status", ""))

        for runner in market.get("runners", []):
            runner_id = str(runner.get("id", ""))
            runner_name = str(runner.get("name", ""))
            status = str(runner.get("status", ""))

            rows.append(
                {
                    "event_id": event_id,
                    "event_name": event_name,
                    "market_id": market_id,
                    "market_name": market_name,
                    "market_type": market_type,
                    "market_status": market_status,
                    "runner_id": runner_id,
                    "runner_name": runner_name,
                    "runner_status": status,
                }
            )

    return pd.DataFrame(rows)

In [ ]:
def retrieve_matchbook_odds(
    event_id: str | int,
    market_filter: str | None = None,
    runner_filter: str | None = None,
    catalogue_df: pd.DataFrame | None = None,
) -> pd.DataFrame:
    """Retrieve matchbook odds ticks filtered by market and runner, ordered chronologically.

    Parameters
    ----------
    event_id : str | int
        The Matchbook event ID.
    market_filter : str, optional
        Market name or ID (e.g., 'Correct Score' or '1X2').
    runner_filter : str, optional
        Runner name or ID (e.g., '1-0' or '1 - 0').
    catalogue_df : pd.DataFrame, optional
        DataFrame returned by get_event_markets_and_runners() to enrich IDs.
    """
    event_id_str = str(event_id)

    where_clauses = [f"cast(event_id as varchar) = '{event_id_str}'"]

    target_market_ids = []
    target_runner_ids = []

    if catalogue_df is not None and not catalogue_df.empty:
        cat = catalogue_df[catalogue_df["event_id"].astype(str) == event_id_str]

        if market_filter:
            m_clean = market_filter.strip().lower()
            m_matches = cat[
                (cat["market_name"].str.lower().str.contains(m_clean, regex=False))
                | (cat["market_type"].str.lower().str.contains(m_clean, regex=False))
                | (cat["market_id"].astype(str) == market_filter.strip())
            ]
            target_market_ids = m_matches["market_id"].unique().tolist()

        if runner_filter:
            r_clean = re.sub(r"\s*-\s*", "-", runner_filter.strip().lower())
            cat_r_norm = cat["runner_name"].str.lower().apply(lambda s: re.sub(r"\s*-\s*", "-", s))
            r_matches = cat[
                cat_r_norm.str.contains(r_clean, regex=False)
                | (cat["runner_id"].astype(str) == runner_filter.strip())
            ]
            target_runner_ids = r_matches["runner_id"].unique().tolist()

    if market_filter:
        if target_market_ids:
            ids_sql = ", ".join(f"'{m}'" for m in target_market_ids)
            where_clauses.append(f"cast(market_id as varchar) IN ({ids_sql})")
        else:
            where_clauses.append(f"cast(market_id as varchar) = '{market_filter.strip()}'")

    if runner_filter:
        if target_runner_ids:
            ids_sql = ", ".join(f"'{r}'" for r in target_runner_ids)
            where_clauses.append(f"cast(runner_id as varchar) IN ({ids_sql})")
        else:
            where_clauses.append(f"cast(runner_id as varchar) = '{runner_filter.strip()}'")

    where_sql = " AND ".join(where_clauses)

    query = f"""
        SELECT
            event_id,
            market_id,
            runner_id,
            ingested_at,
            to_timestamp(ingested_at / 1000.0) as tick_time_utc,
            market_type,
            in_running,
            best_back_price,
            best_back_available,
            best_lay_price,
            best_lay_available,
            wom,
            market_volume,
            runner_volume
        FROM read_parquet('{ODDS_GLOB}', union_by_name=true)
        WHERE {where_sql}
        ORDER BY ingested_at ASC
    """
    try:
        odds_df = con.execute(query).df()
    except Exception as exc:
        print(f"Odds retrieval error (check bronze odds files at {ODDS_GLOB}): {exc}")
        return pd.DataFrame()

    if catalogue_df is not None and not catalogue_df.empty and not odds_df.empty:
        mapping = catalogue_df[
            ["market_id", "market_name", "runner_id", "runner_name"]
        ].drop_duplicates()
        odds_df["market_id"] = odds_df["market_id"].astype(str)
        odds_df["runner_id"] = odds_df["runner_id"].astype(str)
        odds_df = odds_df.merge(mapping, on=["market_id", "runner_id"], how="left")

        front_cols = [
            "tick_time_utc",
            "market_name",
            "runner_name",
            "best_back_price",
            "best_back_available",
            "best_lay_price",
            "best_lay_available",
            "in_running",
        ]
        other_cols = [c for c in odds_df.columns if c not in front_cols]
        odds_df = odds_df[front_cols + other_cols]

    return odds_df

## Step 1: Look Up Match by Home and Away Team Name

Specify the home and away team names (or partial substrings) below to search for matching events.

In [ ]:
HOME_TEAM = "Tampere"
AWAY_TEAM = "OLS"

matches_df = lookup_matchbook_match(home_team=HOME_TEAM, away_team=AWAY_TEAM)
if not matches_df.empty:
    print(f"Found {len(matches_df)} matching event(s):")
    display(matches_df[["event_id", "event_name", "sport_id", "start_utc", "status"]])
else:
    print(f"No events found matching '{HOME_TEAM}' vs '{AWAY_TEAM}'.")
    try:
        sample_query = (
            f"SELECT event_id, event_name, start_utc "
            f"FROM read_parquet('{EVENTS_GLOB}', union_by_name=true) LIMIT 10"
        )
        sample_df = con.execute(sample_query).df()
        print("\nSample events in lake:")
        display(sample_df)
    except Exception as exc:
        print(f"Could not load sample events: {exc}")

## Step 2: Inspect Available Markets & Runners

Select an `event_id` from the search results above to view all markets and runner selections.

In [ ]:
if not matches_df.empty:
    SELECTED_EVENT_ID = str(matches_df.iloc[0]["event_id"])
    raw_event_payload = matches_df.iloc[0]["raw_event"]
    ename = matches_df.iloc[0]["event_name"]
    print(f"Inspecting markets and runners for Event ID {SELECTED_EVENT_ID}: {ename}")

    catalogue_df = get_event_markets_and_runners(raw_event_payload)
    if not catalogue_df.empty:
        m_count = len(catalogue_df["market_id"].unique())
        r_count = len(catalogue_df)
        print(f"\nFound {m_count} market(s) and {r_count} runner selection(s):")
        display(catalogue_df[["market_name", "market_type", "runner_name", "runner_id"]])
    else:
        print(
            "\nNote: raw_event does not contain live market/runner metadata "
            "(common for historical migration rows)."
        )
        catalogue_df = pd.DataFrame()
else:
    SELECTED_EVENT_ID = None
    catalogue_df = pd.DataFrame()

## Step 3: Retrieve & Filter Matchbook Odds Ordered by Event Time

Filter the odds stream by:
- **Market**: e.g. `"Correct Score"` or `"1X2"`
- **Runner**: e.g. `"1-0"` or `"1 - 0"`

The results are ordered chronologically by `ingested_at` (tick timestamp).

In [ ]:
MARKET_FILTER = "Correct Score"  # e.g., "Correct Score" or "1X2" or market_id
RUNNER_FILTER = "1-0"  # e.g., "1-0" or "1 - 0" or runner_id

if SELECTED_EVENT_ID:
    print(
        f"Odds lookup | Event: {SELECTED_EVENT_ID} | "
        f"Market: '{MARKET_FILTER}' | Runner: '{RUNNER_FILTER}'"
    )
    odds_df = retrieve_matchbook_odds(
        event_id=SELECTED_EVENT_ID,
        market_filter=MARKET_FILTER,
        runner_filter=RUNNER_FILTER,
        catalogue_df=catalogue_df,
    )

    if not odds_df.empty:
        print(f"\nRetrieved {len(odds_df):,} ticks ordered by event time:")
        display(odds_df.head(25))
    else:
        msg = f"\nNo ticks matching Market='{MARKET_FILTER}' and Runner='{RUNNER_FILTER}'."
        print(msg)
        print("Showing all available odds ticks for this event across all markets:")
        all_odds_df = retrieve_matchbook_odds(event_id=SELECTED_EVENT_ID, catalogue_df=catalogue_df)
        print(f"Total ticks for event: {len(all_odds_df):,}")
        if not all_odds_df.empty:
            display(all_odds_df.head(15))